In [1]:
import os
import numpy as np
from PIL import Image
from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

IMAGE_SIZE  = 128
BATCH_SIZE  = 32
EPOCHS      = 15
TRAIN_DIR   = "dataset/train"
TEST_DIR    = "dataset/test"
SAVE_PATH   = "model_1_new.keras"
CLASS_NAMES = ["Cyst", "Normal", "Stone", "Tumor"]

In [2]:
train_paths, train_labels = [], []
for label in os.listdir(TRAIN_DIR):
    for img in os.listdir(os.path.join(TRAIN_DIR, label)):
        train_paths.append(os.path.join(TRAIN_DIR, label, img))
        train_labels.append(label)

test_paths, test_labels = [], []
for label in os.listdir(TEST_DIR):
    for img in os.listdir(os.path.join(TEST_DIR, label)):
        test_paths.append(os.path.join(TEST_DIR, label, img))
        test_labels.append(label)

train_paths, train_labels = shuffle(train_paths, train_labels, random_state=42)
test_paths,  test_labels  = shuffle(test_paths,  test_labels,  random_state=42)

print(f"Train: {len(train_paths)} images")
print(f"Test : {len(test_paths)} images")

Train: 4000 images
Test : 800 images


In [3]:
le = LabelEncoder()
le.fit(CLASS_NAMES)
train_labels_enc = to_categorical(le.transform(train_labels), num_classes=4)
test_labels_enc  = to_categorical(le.transform(test_labels),  num_classes=4)
print(f"Classes: {list(le.classes_)}")

def load_images(paths):
    images = []
    for i, p in enumerate(paths):
        img = Image.open(p).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
        images.append(np.array(img, dtype=np.float32) / 255.0)
        if i % 500 == 0:
            print(f"  Loaded {i}/{len(paths)}...")
    return np.array(images)

print("Loading train images...")
X_train = load_images(train_paths)
print("Loading test images...")
X_test  = load_images(test_paths)
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

Classes: [np.str_('Cyst'), np.str_('Normal'), np.str_('Stone'), np.str_('Tumor')]
Loading train images...
  Loaded 0/4000...
  Loaded 500/4000...
  Loaded 1000/4000...
  Loaded 1500/4000...
  Loaded 2000/4000...
  Loaded 2500/4000...
  Loaded 3000/4000...
  Loaded 3500/4000...
Loading test images...
  Loaded 0/800...
  Loaded 500/800...
X_train: (4000, 128, 128, 3)
X_test : (800, 128, 128, 3)


In [4]:
base = VGG16(weights="imagenet", include_top=False,
             input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
base.trainable = False

inputs  = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x       = base(inputs, training=False)
x       = Flatten()(x)          # ✅ no list wrapping
x       = Dense(256, activation="relu")(x)
x       = Dropout(0.5)(x)
outputs = Dense(4, activation="softmax")(x)
model   = Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,813,124 (64.14 MB)

 Trainable params: 2,098,436 (8.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [5]:
callbacks = [
    ModelCheckpoint(SAVE_PATH, monitor="val_accuracy",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=5,
                  restore_best_weights=True, verbose=1),
]

history = model.fit(
    X_train, train_labels_enc,
    validation_data=(X_test, test_labels_enc),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
)

loss, acc = model.evaluate(X_test, test_labels_enc, verbose=0)
print(f"\n✅ Done! Test Accuracy: {acc*100:.2f}%")
print(f"💾 Model saved to: {SAVE_PATH}")

Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4309 - loss: 1.3243
Epoch 1: val_accuracy improved from None to 0.76125, saving model to model_1_new.keras

Epoch 1: finished saving model to model_1_new.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 192s 2s/step - accuracy: 0.5335 - loss: 1.1084 - val_accuracy: 0.7613 - val_loss: 0.7574
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7106 - loss: 0.7621
Epoch 2: val_accuracy improved from 0.76125 to 0.84000, saving model to model_1_new.keras

Epoch 2: finished saving model to model_1_new.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 187s 1s/step - accuracy: 0.7355 - loss: 0.7126 - val_accuracy: 0.8400 - val_loss: 0.5542
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8217 - loss: 0.5513
Epoch 3: val_accuracy improved from 0.84000 to 0.87375, saving model to model_1_new.keras

Epoch 3: finished saving model to model_1_new.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 190s 2s/step - accuracy: 0.8288 - loss: 0.5302 - val_a

In [6]:
# Just confirms the file was saved correctly
import os
print("Model exists:", os.path.exists(SAVE_PATH))
print("Size:", round(os.path.getsize(SAVE_PATH) / 1024 / 1024, 2), "MB")

Model exists: True
Size: 80.24 MB
